# LatentDriver Visualization (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/latentdriver-waymax-experiments/blob/main/notebooks/latentdriver_visualize_colab.ipynb)

This notebook runs a small visualization job and surfaces the generated MP4 or PDF artifacts from the patched upstream simulator.


## What this notebook does

1. sync this repo
2. mount Drive and bind assets/results
3. clone and patch the pinned LatentDriver fork
4. optionally install the heavy runtime
5. run a visualization job on a smoke tier
6. display the generated artifact directly in Colab

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/achyutmorang/latentdriver-waymax-experiments.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/latentdriver-waymax-experiments")


def run(cmd: list[str], cwd: Path | None = None, env: dict[str, str] | None = None, stream: bool = True) -> str:
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    if stream:
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd) if cwd else None,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        lines: list[str] = []
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
        code = proc.wait()
        output = "".join(lines)
        if code != 0:
            raise RuntimeError(f"Command failed with exit code {code}: {printable}")
        return output
    completed = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    return completed.stdout


if REPO_DIR.exists():
    run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
    run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])
else:
    run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)])
repo_rev = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print(json.dumps({"repo_dir": str(REPO_DIR), "repo_rev": repo_rev}, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import json
import os
import sys

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/waymax_research"
os.environ.pop("LATENTDRIVER_RESULTS_ROOT", None)
sys.path.insert(0, str(REPO_DIR / "src"))
from latentdriver_waymax_experiments.colab import bind_drive_layout

binding = bind_drive_layout(DRIVE_ROOT)
print(json.dumps(binding, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import json

run([sys.executable, "scripts/bootstrap_upstream.py"], cwd=REPO_DIR)
lock_path = REPO_DIR / "artifacts" / "locks" / "upstream_bootstrap.json"
print(lock_path.read_text())


In [ ]:
INSTALL_RUNTIME = True

if INSTALL_RUNTIME:
    run([sys.executable, "scripts/setup_colab_runtime.py", "--editable-project"], cwd=REPO_DIR)
else:
    print("Skipping runtime installation. Set INSTALL_RUNTIME=True if this is a fresh Colab session.")


In [ ]:
VIS_MODEL = "latentdriver_t2_j3"
VIS_TIER = "smoke_reactive"
VIS_SEED = 0
VIS_MODE = "video"  # switch to "image" for PDFs
print(json.dumps({
    "model": VIS_MODEL,
    "tier": VIS_TIER,
    "seed": VIS_SEED,
    "vis": VIS_MODE,
}, indent=2, sort_keys=True))


In [ ]:
import json

output = run([
    sys.executable,
    "scripts/run_visualization.py",
    "--model",
    VIS_MODEL,
    "--tier",
    VIS_TIER,
    "--seed",
    str(VIS_SEED),
    "--vis",
    VIS_MODE,
], cwd=REPO_DIR)
payload = json.loads(output)
print(json.dumps(payload, indent=2, sort_keys=True))


In [ ]:
from IPython.display import Video, display

vis_root = Path(payload["vis_dir"])
mp4s = sorted(vis_root.glob("**/*.mp4"))
pdfs = sorted(vis_root.glob("**/*.pdf"))
print(json.dumps({
    "vis_root": str(vis_root),
    "mp4_count": len(mp4s),
    "pdf_count": len(pdfs),
}, indent=2, sort_keys=True))

if mp4s:
    print(f"Displaying {mp4s[0]}")
    display(Video(str(mp4s[0]), embed=True))
elif pdfs:
    print("Generated PDF files:")
    for path in pdfs[:10]:
        print(path)
else:
    print("No visualization files found. Inspect stderr.log in the run directory.")


## Next step

After you have one visualization artifact, move back to the public evaluation notebook and run the standardized suite on the full validation artifacts.